# Quickstart: Using NexaMem with Azure Redis Cache

This notebook demonstrates how to configure and use NexaMem with Azure Redis Cache for chatbot history management. It uses the local development version of NexaMem.

Import NexaMem

In [1]:
import sys
import os
import subprocess

# Add the src directory to the Python path
# This is so we can import the local nexamem package
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

# Install the redis package
subprocess.run([sys.executable, "-m", "pip", "install", "redis"], check=True)

CompletedProcess(args=['c:\\Users\\chadada\\source\\repos\\nexamem\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'redis'], returncode=0)

Configure and create history

In [ ]:
from nexamem.config import initialize
from nexamem.history import ChatHistory
import json

# 1. Configure NexaMem to use Azure Redis Cache
# Replace with your Azure Redis Cache endpoint and primary key
redis_config = {
    "host": "your-custom-hostname.redis.cache.windows.net",
    "port": 6380,
    "password": "your-access-key",
    "ssl": True
}

initialize({
    "history_storage": "redis",
    "redis_config": redis_config,
    "debug": True
})

# 2. Create a ChatHistory instance and set session context
chat_history = ChatHistory()
chat_history.set_session_context(
    session_id="session-12345",
    agent_id="demo-agent-007",
    user_id="user-jane-doe",
    env="dev"
)

# 3. Add messages to the chat history with more properties
chat_history.add_user_message(
    "Hello, NexaMem!",
    tokens=5,
    data=json.dumps({"source": "notebook"})
)
chat_history.add_assistant_message(
    "Hello! How can I assist you today?",
    tokens=10
)

# 4. Retrieve and print the chat history, showing the new properties
history = chat_history.get_history()
for message in history:
    print(
        f"Role: {message['role']}, Content: \"{message['content']}\"\n"
        f"  Agent: {message['agent_id']}, User: {message['user_id']}, Session: {message['session_id']}\n"
        f"  Env: {message['env']}, Tokens: {message['tokens']}, Data: {message['data']}\n"
        f"  Timestamp: {message['timestamp']}\n"
    )

# 5. Clear the history for this session
# chat_history.clear()
# print("\nChat history cleared.")

  NexaMem: AI Memory Manager
  Successfully initialized!
  https://github.com/microsoft/nexamem
[DEBUG] Initialized Redis connection: {'host': 'nexamem-test.redis.cache.windows.net', 'port': 6380, 'ssl': True, 'db': 0, 'connected': True, 'use_fakeredis': False, 'is_azure': False}
[DEBUG] Loaded 0 messages from Redis
[DEBUG] ChatHistory initialized with storage='redis'
[DEBUG] Set session context: {'agent_id': 'demo-agent-007', 'user_id': 'user-jane-doe', 'session_id': 'session-12345', 'env': 'dev'}
[DEBUG] Saved message to Redis: nexamem:chat_history:80962f3f-52ef-49a3-accf-e17505c4187d
[DEBUG] Added message: role='user' content='Hello, NexaMem!' agent_id='demo-agent-007' session_id='session-12345'
[DEBUG] Saved message to Redis: nexamem:chat_history:0dddfb14-1ed7-48ae-b1e3-db3f529cb077
[DEBUG] Added message: role='assistant' content='Hello! How can I assist you today?' agent_id='demo-agent-007' session_id='session-12345'
[DEBUG] Loaded 2 messages from Redis
[DEBUG] get_history called.

### Using AIMemory with YAML Configuration

The `AIMemory` class provides a more advanced, policy-driven interface for managing conversational memory. It uses a YAML file to configure different memory "channels," each with its own TTL (Time To Live) and policies.

First, let's ensure a `channels.yaml` file exists.

In [ ]:
import os
from nexamem.aimemory import AIMemory
from nexamem.memory_scope import MemoryScope

# Define the YAML configuration as a string (using correct dictionary format)
channels_yaml_content = """
channels:
  general:
    ttl_sec: 86400  # 24 hours
    encrypt: false

  customer_support:
    ttl_sec: 604800  # 7 days
    encrypt: false
"""

# Write the YAML content to a file
channels_yaml_path = "channels.yaml"
with open(channels_yaml_path, "w") as f:
    f.write(channels_yaml_content)

print(f"Created '{channels_yaml_path}' for AIMemory configuration.")

# 2. Initialize AIMemory with Redis config and channels file
# We need to unpack the redis_config dictionary into individual parameters
ai_memory = AIMemory(
    redis_host=redis_config["host"],
    redis_port=redis_config["port"],
    redis_password=redis_config["password"],
    redis_ssl=redis_config["ssl"],
    channels_yaml=channels_yaml_path
)

# 3. Define a memory scope for the interaction (including required agent_id)
scope = MemoryScope(
    agent_id="demo-agent-007",
    user_id="user-jane-doe",
    session_id="session-12345"
)

# 4. Write a message to the 'general' channel
print("\nWriting to 'general' channel...")
ai_memory.write(scope, "general", "This is a test message in the general channel.")

# 5. Read the memory from the 'general' channel
print("Reading from 'general' channel...")
messages = ai_memory.read(scope, "general")

# 6. Print the retrieved messages
for msg in messages:
    print(f"  - Retrieved: {msg}")

# 7. Clean up the created YAML file
# os.remove(channels_yaml_path)
# print(f"\nCleaned up '{channels_yaml_path}'.")

Created 'channels.yaml' for AIMemory configuration.

Writing to 'general' channel...
Reading from 'general' channel...

Writing to 'general' channel...
Reading from 'general' channel...
  - Retrieved: ['This is a test message in the general channel.']
  - Retrieved: {'total_messages': 1, 'total_bytes': 46, 'truncated': False}

Cleaned up 'channels.yaml'.
  - Retrieved: ['This is a test message in the general channel.']
  - Retrieved: {'total_messages': 1, 'total_bytes': 46, 'truncated': False}

Cleaned up 'channels.yaml'.
